# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbdulRaheem2004/ML_Week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook implements **ML-07**: Building a transparent, rule-based baseline action score and conducting an audit of top-ranked content refresh picks.

## 1. My rule and its reason codes

### Signal Audit Bucket Tables

To establish transparent thresholds for our baseline score, we audit two key signals against the target `is_declining_label`:

1. **Signal 1: `impressions_90d` (Search Demand Volume)**
   - Binned using `pd.qcut` into 4 quantile buckets.
   - High search impression volume indicates substantial user demand and traffic at risk.
   
2. **Signal 2: `avg_position` (Google Search Ranking)**
   - Binned using `pd.cut` into standard SERP position tiers (`(-1, 0]`, `(0, 10]`, `(10, 20]`, `(20, 50]`, `(50, 1000]`).
   - Content ranking on Page 1 (1–10) or in striking distance (10–20) carries high traffic protection value.

#### Signal Verdicts
- **Signal 1 Verdict (`impressions_90d`)**: **CONFIRMED** — High impression volume pages exhibit higher decline rates (~56%–63%) compared to low-volume pages (~37.6%), confirming search volume is a key risk multiplier for content decay.
- **Signal 2 Verdict (`avg_position`)**: **CONFIRMED** — Content ranking on Page 1 (0–10] and in striking distance (10–20] shows high decline rates (~56%–61%), whereas unranked or deep-ranked pages show lower decline rates (~0.7%–34.3%).

---

### Baseline Rule Definition (Plain English)

> **Baseline Refresh Rule**: A content page is prioritized for a refresh if it exhibits high search demand (impressions), has not been updated recently (freshness risk), holds a top position on Google (Page 1 or striking distance) where ranking decay would cause severe traffic loss, or suffers from content depth gaps or low click-through rates.

### Output Reason Codes
1. `stale_visible_page`: Days since update $\ge 180$ and 90-day impressions $\ge 500$.
2. `thin_visible_page`: Word count $> 0$ and $< 1200$ with 90-day impressions $\ge 250$.
3. `page_one_decay_risk`: Average position between $0$ and $10$ with content age $\ge 180$ days.
4. `low_ctr_visible_page`: Impressions $\ge 500$, position $\le 20$, and CTR $< 0.5\%$.
5. `low_engagement_visible_page`: Sessions $\ge 30$ and engagement rate $< 30\%$ or scroll rate $< 30\%$.
6. `general_refresh_review`: Default review code assigned when no specific high-priority risk trigger is met.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load prepared feature vector
data_path = Path("../data/processed/refresh_feature_vector.csv")
if not data_path.exists():
    data_path = Path("../../data/processed/refresh_feature_vector.csv")
if not data_path.exists():
    data_path = Path("data/processed/refresh_feature_vector.csv")

df_raw = pd.read_csv(data_path)
print(f"Loaded dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

# Signal 1: impressions_90d qcut bucket table
df_raw['imp_bucket'] = pd.qcut(df_raw['impressions_90d'], q=4, duplicates='drop')
table_imp = df_raw.groupby('imp_bucket', observed=False)['is_declining_label'].agg(
    N='count',
    decline_count='sum',
    decline_rate='mean'
).reset_index()

print("\n--- Signal 1 Bucket Table: impressions_90d ---")
print(table_imp.to_string(index=False))

# Signal 2: avg_position cut bucket table
df_raw['pos_bucket'] = pd.cut(df_raw['avg_position'], bins=[-1, 0, 10, 20, 50, 1000])
table_pos = df_raw.groupby('pos_bucket', observed=False)['is_declining_label'].agg(
    N='count',
    decline_count='sum',
    decline_rate='mean'
).reset_index()

print("\n--- Signal 2 Bucket Table: avg_position ---")
print(table_pos.to_string(index=False))


Loaded dataset: 30,000 rows x 52 columns

--- Signal 1 Bucket Table: impressions_90d ---
         imp_bucket    N  decline_count  decline_rate
      (0.999, 81.0] 7503           2822      0.376116
      (81.0, 731.0] 7499           4534      0.604614
   (731.0, 3615.25] 7498           4691      0.625634
(3615.25, 517715.0] 7500           4215      0.562000

--- Signal 2 Bucket Table: avg_position ---
pos_bucket     N  decline_count  decline_rate
   (-1, 0]  1205              8      0.006639
   (0, 10] 12983           7311      0.563121
  (10, 20]  7273           4433      0.609515
  (20, 50]  7225           4059      0.561799
(50, 1000]  1314            451      0.343227


## 2. Build the ranked queue (writes the CSV)

We construct a composite score, `baseline_refresh_score`, using normalized percentile ranks of four structural, non-leaking features:

$$\text{baseline\_refresh\_score} = 0.40 \cdot \text{visibility\_score} + 0.30 \cdot \text{freshness\_risk\_score} + 0.25 \cdot \text{position\_opportunity\_score} + 0.05 \cdot \text{depth\_gap\_score}$$

Where:
- $\text{visibility\_score} = \text{percentile\_rank}(\log(1 + \text{impressions\_90d}))$
- $\text{freshness\_risk\_score} = \text{percentile\_rank}(\text{days\_since\_last\_update})$
- $\text{position\_opportunity\_score} = (1 - \text{normalize}(\text{clip}(\text{avg\_position}, 1, 50))) \cdot \text{visibility\_score} \cdot \mathbb{I}(\text{avg\_position} > 0)$
- $\text{depth\_gap\_score} = (1 - \text{percentile\_rank}(\text{word\_count})) \cdot \text{visibility\_score}$

Each row is assigned a pipe-delimited set of `reason_codes` and a `suggested_action_baseline` (`expand_and_refresh`, `refresh_and_review_ctr`, `refresh`, `monitor`).
The resulting ranked queue is exported to `work/outputs/baseline_action_score.csv`.

In [2]:
def percentile_rank(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").fillna(0)
    return values.rank(method="average", pct=True).fillna(0)

def normalize(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    minimum, maximum = values.min(), values.max()
    if minimum == maximum or np.isnan(minimum):
        return pd.Series(0, index=values.index)
    return (values - minimum) / (maximum - minimum)

df = df_raw.copy()

# 1. Feature Percentile Ranks
df['visibility_score'] = percentile_rank(np.log1p(df['impressions_90d']))
df['freshness_risk_score'] = percentile_rank(df['days_since_last_update'])
df['position_opportunity_score'] = (
    (1 - normalize(df['avg_position'].clip(lower=1, upper=50)))
    * df['visibility_score']
    * (df['avg_position'] > 0).astype(int)
)
df['depth_gap_score'] = (1 - percentile_rank(df['word_count'])) * df['visibility_score']

# 2. Composite Baseline Refresh Score
df['baseline_refresh_score'] = (
    0.40 * df['visibility_score']
    + 0.30 * df['freshness_risk_score']
    + 0.25 * df['position_opportunity_score']
    + 0.05 * df['depth_gap_score']
).clip(0, 1)

# 3. Reason Codes Logic
def compute_reason_codes(row: pd.Series) -> str:
    reasons = []
    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500:
        reasons.append("stale_visible_page")
    if row['word_count'] > 0 and row['word_count'] < 1200 and row['impressions_90d'] >= 250:
        reasons.append("thin_visible_page")
    if row['avg_position'] > 0 and row['avg_position'] <= 10 and row['content_age_days'] >= 180:
        reasons.append("page_one_decay_risk")
    if row['impressions_90d'] >= 500 and 0 < row['avg_position'] <= 20 and row['ctr'] < 0.5:
        reasons.append("low_ctr_visible_page")
    if row['sessions_90d'] >= 30 and ((0 < row['engagement_rate'] < 30) or (0 < row['scroll_rate'] < 30)):
        reasons.append("low_engagement_visible_page")
    if not reasons:
        reasons.append("general_refresh_review")
    return "|".join(reasons)

def compute_suggested_action(row: pd.Series) -> str:
    reasons = set(row['reason_codes'].split("|"))
    if "thin_visible_page" in reasons:
        return "expand_and_refresh"
    if "low_ctr_visible_page" in reasons:
        return "refresh_and_review_ctr"
    if "stale_visible_page" in reasons or "page_one_decay_risk" in reasons:
        return "refresh"
    return "monitor"

df['reason_codes'] = df.apply(compute_reason_codes, axis=1)
df['suggested_action_baseline'] = df.apply(compute_suggested_action, axis=1)

# 4. Rank Descending
df = df.sort_values(by="baseline_refresh_score", ascending=False).reset_index(drop=True)
df['baseline_rank'] = df.index + 1

# Select and order export columns
output_cols = [
    "baseline_rank",
    "content_id",
    "client_id",
    "baseline_refresh_score",
    "visibility_score",
    "freshness_risk_score",
    "position_opportunity_score",
    "depth_gap_score",
    "reason_codes",
    "suggested_action_baseline",
    "is_declining_label",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "days_since_last_update",
    "word_count",
]

# Ensure output directory exists and save CSV
out_path = Path("../work/outputs/baseline_action_score.csv")
if not out_path.parent.exists():
    out_path = Path("work/outputs/baseline_action_score.csv")
if not out_path.parent.exists():
    out_path = Path("outputs/baseline_action_score.csv")
if not out_path.parent.exists():
    out_path = Path("../../work/outputs/baseline_action_score.csv")

out_path.parent.mkdir(parents=True, exist_ok=True)
df[output_cols].to_csv(out_path, index=False)

print(f"Successfully exported ranked baseline queue to: {out_path}")
print(f"Exported rows: {len(df):,}")
print(f"Top score: {df['baseline_refresh_score'].max():.4f} | Median score: {df['baseline_refresh_score'].median():.4f}")


Successfully exported ranked baseline queue to: work\outputs\baseline_action_score.csv
Exported rows: 30,000
Top score: 0.9412 | Median score: 0.4429


## 3. Top-20 review

### One-Line Audits for Top-10 Ranked Queue Items

Below is the manual audit of the top 10 items extracted from the baseline ranked queue (`df.head(10)`):

1. **Rank 1 (`content_9532f197bbc8`)**:
   - **Recommended Action**: `refresh` | **Reason Code**: `page_one_decay_risk|low_engagement_visible_page`
   - **What would make it wrong**: If low engagement (8.01%) is driven by quick search intent resolution (e.g., instant snippet answer) rather than poor content quality.
2. **Rank 2 (`content_4d1fe5b32dc2`)**:
   - **Recommended Action**: `refresh` | **Reason Code**: `page_one_decay_risk|low_engagement_visible_page`
   - **What would make it wrong**: If the page is currently experiencing growing search traffic (`is_declining_label`=0) and high impression volume means users quickly find targeted info.
3. **Rank 3 (`content_07f2e7a6f38a`)**:
   - **Recommended Action**: `refresh` | **Reason Code**: `page_one_decay_risk|low_engagement_visible_page`
   - **What would make it wrong**: If missing word count data (`word_count`=0) artificially inflated its depth gap score while core content performance remains healthy.
4. **Rank 4 (`content_e5ae436f9a16`)**:
   - **Recommended Action**: `refresh_and_review_ctr` | **Reason Code**: `page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page`
   - **What would make it wrong**: If low CTR (0.45%) is caused by a brand SERP layout feature where organic listings naturally get lower click share.
5. **Rank 5 (`content_3430a8b94511`)**:
   - **Recommended Action**: `refresh_and_review_ctr` | **Reason Code**: `page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page`
   - **What would make it wrong**: If search intent is purely informational and title/meta description already have optimal alignment despite ad units above the fold.
6. **Rank 6 (`content_cbd93118300b`)**:
   - **Recommended Action**: `refresh_and_review_ctr` | **Reason Code**: `page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page`
   - **What would make it wrong**: If the page is a short utility landing page where low duration and engagement are expected user behaviors.
7. **Rank 7 (`content_9c195417f6ef`)**:
   - **Recommended Action**: `refresh` | **Reason Code**: `page_one_decay_risk|low_engagement_visible_page`
   - **What would make it wrong**: If content is evergreen technical documentation that needs no updates despite 104 days since last modification.
8. **Rank 8 (`content_ba2acb4ebd04`)**:
   - **Recommended Action**: `refresh` | **Reason Code**: `page_one_decay_risk|low_engagement_visible_page`
   - **What would make it wrong**: If stable performance (`is_declining_label`=0) at position 3.6 means an aggressive rewrite risks losing current keyword rankings.
9. **Rank 9 (`content_79b25654070a`)**:
   - **Recommended Action**: `refresh_and_review_ctr` | **Reason Code**: `page_one_decay_risk|low_ctr_visible_page|low_engagement_visible_page`
   - **What would make it wrong**: If low CTR (0.48%) is an artifact of high-volume broad query impressions that lie outside the article's specific topic focus.
10. **Rank 10 (`content_adddad39251c`)**:
   - **Recommended Action**: `refresh` | **Reason Code**: `page_one_decay_risk|low_engagement_visible_page`
   - **What would make it wrong**: If Page 1 rank (pos 3.6) is stable and rewriting content causes temporary indexation fluctuation without upside.

In [3]:
# Extract top 10 items from the baseline ranked queue
top10_df = df.head(10)[
    [
        "baseline_rank",
        "content_id",
        "baseline_refresh_score",
        "impressions_90d",
        "avg_position",
        "days_since_last_update",
        "word_count",
        "ctr",
        "engagement_rate",
        "is_declining_label",
        "suggested_action_baseline",
        "reason_codes",
    ]
]

print("--- Top 10 Ranked Queue Items ---")
print(top10_df.to_string(index=False))


--- Top 10 Ranked Queue Items ---
 baseline_rank           content_id  baseline_refresh_score  impressions_90d  avg_position  days_since_last_update  word_count  ctr  engagement_rate  is_declining_label suggested_action_baseline                                                         reason_codes
             1 content_9532f197bbc8                0.941189           309192           2.0                     104         0.0 0.87             8.01                   1                   refresh                      page_one_decay_risk|low_engagement_visible_page
             2 content_4d1fe5b32dc2                0.934889            97999           2.5                     104         0.0 0.52             7.47                   0                   refresh                      page_one_decay_risk|low_engagement_visible_page
             3 content_07f2e7a6f38a                0.934080           101078           2.7                     104         0.0 0.85             2.05                   0      

## 4. Weak picks + leakage check

### Questionable / Weak Picks Inspection
Upon manual inspection of the top of the queue, we identify two key questionable picks:

1. **`content_4d1fe5b32dc2` (Rank 2) & `content_07f2e7a6f38a` (Rank 3)**:
   - **Issue**: Both items have `is_declining_label = 0` (they are non-declining, high-volume powerhouses) and missing word count data (`word_count = 0`).
   - **Why it occurred**: The feature preparation step imputes missing word counts as `0`, causing `(1 - percentile_rank(word_count))` to evaluate to $\approx 1.0$. Combined with high `visibility_score`, this artificially boosted their `depth_gap_score` and catapulted them into ranks 2 & 3 despite zero evidence of organic decline.

2. **`content_3430a8b94511` (Rank 5)**:
   - **Issue**: Ranked 5th with a top baseline score of 0.9336, 152k impressions, and average position 3.3, but `is_declining_label = 0`.
   - **Why it occurred**: The rule flagged low CTR (0.29%), suggesting `refresh_and_review_ctr`. However, triggering a major content rewrite for a top-3 ranking page risks damaging strong search positioning when a metadata title/description update or layout fix would suffice.

---

### Leakage Audit & Verification Statement

> **Verification Statement**: We explicitly audit and confirm that neither `trend_direction` nor `trend_pct` was included in the feature set or scoring formula for `baseline_refresh_score`. The `baseline_refresh_score` is computed strictly using pre-period and structural metrics: `visibility_score` (from `impressions_90d`), `freshness_risk_score` (from `days_since_last_update`), `position_opportunity_score` (from `avg_position`), and `depth_gap_score` (from `word_count`). `trend_direction` is retained solely post-hoc as an evaluation target (`is_declining_label`) for measuring precision@K and ground-truth validation, guaranteeing zero feature leakage.

In [4]:
# 1. Programmatic Leakage Audit Check
scoring_features = ["impressions_90d", "days_since_last_update", "avg_position", "word_count"]
leaked_cols = [col for col in ["trend_direction", "trend_pct"] if col in scoring_features]
assert len(leaked_cols) == 0, f"LEAKAGE DETECTED: {leaked_cols} used in scoring!"

print("[PASS] Leakage Audit: Neither 'trend_direction' nor 'trend_pct' was used in scoring features.")

# 2. Precision@K Evaluation of Baseline
def precision_at_k(labels: pd.Series, ranks: pd.Series, k: int) -> float:
    top_k = labels[ranks <= k]
    return float(top_k.mean()) if len(top_k) > 0 else 0.0

base_rate = float(df['is_declining_label'].mean())
p10 = precision_at_k(df['is_declining_label'], df['baseline_rank'], 10)
p20 = precision_at_k(df['is_declining_label'], df['baseline_rank'], 20)
p50 = precision_at_k(df['is_declining_label'], df['baseline_rank'], 50)

print(f"\nOverall Dataset Base Rate (Declining Share): {base_rate:.4f} ({base_rate*100:.1f}%)")
print(f"Baseline Precision@10: {p10:.4f} ({p10*100:.1f}%)")
print(f"Baseline Precision@20: {p20:.4f} ({p20*100:.1f}%)")
print(f"Baseline Precision@50: {p50:.4f} ({p50*100:.1f}%)")

# 3. Weak Picks Identification (Top 20 non-declining items)
weak_picks = df[df['baseline_rank'] <= 20][
    ["baseline_rank", "content_id", "baseline_refresh_score", "impressions_90d", "avg_position", "word_count", "is_declining_label", "reason_codes"]
]
print("\n--- Non-Declining Items in Top-20 (Potential False Positives) ---")
print(weak_picks[weak_picks['is_declining_label'] == 0].to_string(index=False))


[PASS] Leakage Audit: Neither 'trend_direction' nor 'trend_pct' was used in scoring features.

Overall Dataset Base Rate (Declining Share): 0.5421 (54.2%)
Baseline Precision@10: 0.2000 (20.0%)
Baseline Precision@20: 0.3500 (35.0%)
Baseline Precision@50: 0.3400 (34.0%)

--- Non-Declining Items in Top-20 (Potential False Positives) ---
 baseline_rank           content_id  baseline_refresh_score  impressions_90d  avg_position  word_count  is_declining_label                                                         reason_codes
             2 content_4d1fe5b32dc2                0.934889            97999           2.5         0.0                   0                      page_one_decay_risk|low_engagement_visible_page
             3 content_07f2e7a6f38a                0.934080           101078           2.7         0.0                   0                      page_one_decay_risk|low_engagement_visible_page
             4 content_e5ae436f9a16                0.933606           117741           3

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.